# Go2 parallel swing-up and front-foot catching
This notebook runs a bounded CPU MuJoCo pilot with one shared actor per condition across all its environments. It benchmarks 8/16/32/64 environments, chooses 16 or 32 using measured throughput, and trains matched A/B/C conditions for 10 updates on two seeds. W&B is disabled. Existing checkpoints are never overwritten.

Upload `front_catch_anticipatory_molab_20261112_v3.zip` next to this notebook. Python 3.11 is required. This notebook was prepared and tested through its underlying commands locally; no remote Molab session was launched during preparation.
This version selects `--suite anticipatory_v1`. The 35-test suite and two-update local smoke validate plumbing and safety, not learned catching. See `docs/front_catch_anticipatory.md`. No cloud run has been launched.


In [ ]:
from pathlib import Path
import os, sys, json, zipfile, hashlib, subprocess, time, uuid
assert sys.version_info[:2] == (3, 11), "Select a Python 3.11 runtime."
UPLOAD_ROOT = globals().get("UPLOAD_ROOT", Path.cwd())
BUNDLE = UPLOAD_ROOT / "front_catch_anticipatory_molab_20261112_v3.zip"
WORKSPACE = UPLOAD_ROOT / "front_catch_anticipatory_workspace_v1"
RUN_DIR = Path("output/front_catch_anticipatory_molab_v1")
PREPARATION = Path("output/front_catch_preparation_20261112_v3")
UPDATES, HORIZON, SEEDS = 10, 32, [42, 43]
MAX_TRAIN_SECONDS = 1200  # Leaves time for reports/backups; resume in a later session.
RESUME_LATEST = True
os.environ.update(WANDB_MODE="disabled", WANDB_DISABLED="true", OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", MPLCONFIGDIR="/tmp/front-catch-mpl", XDG_CACHE_HOME="/tmp/front-catch-cache")
if not WORKSPACE.exists():
    WORKSPACE.mkdir()
    with zipfile.ZipFile(BUNDLE) as bundle:
        bundle.extractall(WORKSPACE)
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
manifest = json.loads(Path("front_catch_bundle_manifest.json").read_text())
for name, digest in manifest.items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == digest, f"Changed bundle source/input: {name}"
print("Verified source and preserved initial checkpoint.")

In [ ]:
# Retain an existing working torch installation; install CPU torch only if absent.
import importlib.util
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "experiments/front_catch/requirements.txt"], check=True)
run_env = dict(os.environ)
run_env.pop("PYTHONPATH", None)
subprocess.run([sys.executable, "-c", "import mujoco,torch,numpy; print('MuJoCo',mujoco.__version__,'torch',torch.__version__,'NumPy',numpy.__version__); assert mujoco.__version__=='3.11.0'"], env=run_env, check=True)
# A legacy morphology test records git HEAD. The ZIP contains source, not history.
# Record a new local archive snapshot, not a claim to restore the original history.
if not Path(".git").exists():
    subprocess.run(["git", "init", "-q"], env=run_env, check=True)
    subprocess.run(["git", "add", "--", "experiments", "scripts", "tests", "front_catch_bundle_manifest.json"], env=run_env, check=True)
    subprocess.run(["git", "-c", "user.name=Molab Archive Snapshot", "-c", "user.email=archive@invalid", "commit", "-qm", "Snapshot hash-verified front-catch source archive"], env=run_env, check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/unit_tests/test_front_catch.py", "tests/unit_tests/test_front_catch_anticipatory.py", "tests/unit_tests/test_momentum_phase.py", "tests/unit_tests/test_com_energy_phase.py", "tests/unit_tests/test_physical_morphology_ppo.py", "-q"], env=run_env, check=True)

## Benchmark this Molab host
Throughput is controller transitions per second (10 native physics substeps each). Memory is aggregate parent/worker RSS, which may count shared pages more than once. The selector chooses the smaller 16/32 count if performance is within 5%; it never selects 64 for this pilot. A conservative worker estimate accounts for CPU and cgroup memory limits. Keep the benchmark isolated from other training jobs.

In [ ]:
import math
cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
quota = Path("/sys/fs/cgroup/cpu.max")
if quota.exists():
    q, period = quota.read_text().split()
    if q != "max": cpus = min(cpus, max(1, math.ceil(int(q)/int(period))))
workers = min(4, cpus)
limit = Path("/sys/fs/cgroup/memory.max")
if limit.exists() and limit.read_text().strip() != "max":
    gib = int(limit.read_text().strip()) / 1024**3
    workers = min(workers, max(0, int((0.75*gib-1.6)/0.8)))
stamp = time.strftime("%Y%m%d_%H%M%S", time.gmtime()) + "_" + uuid.uuid4().hex[:6]
BENCHMARK = Path("output") / ("front_catch_benchmark_" + stamp)
subprocess.run([sys.executable, "scripts/benchmark_front_catch.py", "--suite", "anticipatory_v1", "--preparation", str(PREPARATION), "--output", str(BENCHMARK), "--workers", str(workers), "--counts", "8", "16", "32", "64"], env=run_env, check=True)
selection = json.loads((BENCHMARK/"selection.json").read_text())
import pandas as pd
from IPython.display import display
print(selection)
display(pd.read_csv(BENCHMARK/"benchmark.csv"))
print("Planned fresh-run interactions:", selection["num_envs"]*HORIZON*UPDATES*3*len(SEEDS))

## Train or resume, then always back up
A retains the original fixed-support baseline. B adds earlier predicted support placement during the rising swing; C adds coordinated rear-hip/knee momentum redistribution. B and C share the same contact-transition gates and allow staggered front steps or slides. The physical model and friction are identical. All actors retain the successful kick-up BC regularizer. Standing and intermediate-start success are reported separately.

Resumption keeps the original environment count, rollout horizon, seeds and total update budget. The worker count may change with the host. Checkpoints include all policies, optimizer/RNG states, world/contact/phase states and committed metrics. Old checkpoints and uncommitted crash logs remain intact. Only committed history is used for reports. No run here approaches a million transitions.

In [ ]:
import tarfile
from IPython.display import FileLink
REPORT = Path("output") / ("front_catch_report_" + stamp)
BACKUP = Path("output") / ("front_catch_backup_" + stamp + ".tar.gz")
exit_code = 0
PRESERVE_DIRS = []
# Each periodic archive contains one immutable full checkpoint and its committed
# traces. Original older checkpoints remain in RUN_DIR and in the final backup.
def periodic_backup(checkpoint):
    archive_path = Path("output") / ("front_catch_recovery_" + stamp + "_" + checkpoint.stem + ".tar.gz")
    result = subprocess.run([sys.executable, "scripts/backup_front_catch.py", str(RUN_DIR), "--checkpoint", str(checkpoint), "--output", str(archive_path), "--bundle", str(BUNDLE)], env=run_env, capture_output=True, text=True, check=True)
    print(result.stdout)
    display(FileLink(str(archive_path)))
try:
    checkpoints = sorted(RUN_DIR.glob("checkpoint_*.pt")) if RUN_DIR.exists() else []
    if RUN_DIR.exists() and not checkpoints:
        PRESERVE_DIRS.append(RUN_DIR)
        RUN_DIR = RUN_DIR.with_name(RUN_DIR.name + "_restart_" + stamp)
        print("No complete checkpoint; preserving the old attempt and using", RUN_DIR)
    command = [sys.executable, "scripts/run_front_catch.py", "--suite", "anticipatory_v1", "--output", str(RUN_DIR), "--workers", str(selection["workers"]), "--max-seconds", str(MAX_TRAIN_SECONDS)]
    completed = False
    if checkpoints:
        assert RESUME_LATEST, "Choose a fresh RUN_DIR to avoid overwriting the old run."
        protocol = json.loads((RUN_DIR/"protocol.json").read_text())
        assert protocol.get("comparison_suite") == "anticipatory_v1", "Use a fresh RUN_DIR for the new objectives."
        completed = int(checkpoints[-1].stem.rsplit("_",1)[1]) >= protocol["updates"]
        command += ["--resume", str(checkpoints[-1])]
        print("Resuming with original", protocol["num_envs"], "environments.")
    else:
        command += ["--preparation", str(PREPARATION), "--benchmark", str(BENCHMARK/"selection.json"), "--updates", str(UPDATES), "--horizon", str(HORIZON), "--seeds"] + [str(s) for s in SEEDS]
    if not completed:
        process = subprocess.Popen(command, env=run_env)
        backed_up = set()
        try:
            while process.poll() is None:
                for checkpoint in sorted(RUN_DIR.glob("checkpoint_*.pt")):
                    if checkpoint not in backed_up:
                        periodic_backup(checkpoint)
                        backed_up.add(checkpoint)
                time.sleep(5)
            exit_code = process.returncode
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait(timeout=60)
    if list(RUN_DIR.glob("checkpoint_*.pt")):
        subprocess.run([sys.executable, "scripts/report_front_catch.py", str(RUN_DIR), "--output", str(REPORT), "--benchmark", str(BENCHMARK)], env=run_env, check=True)
        subprocess.run([sys.executable, "scripts/report_front_catch_timing.py", str(RUN_DIR), "--output", str(REPORT/"anticipatory_timing")], env=run_env, check=True)
        print(json.loads((REPORT/"completion.json").read_text()))
finally:
    if list(RUN_DIR.glob("checkpoint_*.pt")):
        backup_command = [sys.executable, "scripts/backup_front_catch.py", str(RUN_DIR), "--all", "--output", str(BACKUP), "--bundle", str(BUNDLE)]
        for directory in PRESERVE_DIRS + [BENCHMARK, REPORT]:
            if directory.exists(): backup_command += ["--extra", str(directory)]
        subprocess.run(backup_command, env=run_env, check=True)
    else:
        # Preserve diagnostics even if failure happened before the first checkpoint.
        with tarfile.open(BACKUP, "x:gz") as archive:
            for directory in PRESERVE_DIRS + [RUN_DIR, BENCHMARK, REPORT]:
                if directory.exists(): archive.add(directory)
            archive.add(BUNDLE, arcname=BUNDLE.name)
    print("Backup:", BACKUP, "SHA256:", hashlib.sha256(BACKUP.read_bytes()).hexdigest())
    display(FileLink(str(BACKUP)))
if exit_code:
    raise RuntimeError("Training stopped with an error. Backup preserved; inspect logs and resume from the latest complete checkpoint.")

In [ ]:
from IPython.display import Image
if (REPORT/"standing_summary.csv").exists():
    display(pd.read_csv(REPORT/"standing_summary.csv"))
    display(pd.read_csv(REPORT/"intermediate_summary.csv"))
    display(Image(filename=str(REPORT/"standing_learning.png")))
    display(Image(filename=str(REPORT/"foot_placement_and_collisions.png")))
print("Download the backup before ending this Molab session. It contains the source bundle and all completed checkpoints.")